<a href="https://colab.research.google.com/github/72marineblue-prog/github-basic-kadai/blob/main/Python%E3%81%A7%E6%A5%AD%E5%8B%99%E3%82%92%E8%87%AA%E5%8B%95%E5%8C%96%E3%81%99%E3%82%8B%E3%82%B7%E3%82%B9%E3%83%86%E3%83%A0%E3%82%92%E9%96%8B%E7%99%BA%E3%81%97%E3%81%A6%E3%81%BF%E3%82%88%E3%81%86.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [52]:
# =========================================
# ライブラリ
# =========================================

import os
import pandas as pd
from datetime import datetime

# =========================================
# フォルダ作成
# =========================================

os.makedirs("project/samples/order_new", exist_ok=True)

# =========================================
# 注文Excel作成
# =========================================

order_A = pd.DataFrame({
    "野菜": ["キャベツ", "にんじん"],
    "注文数": [30, 20]
})

order_B = pd.DataFrame({
    "野菜": ["キャベツ", "玉ねぎ"],
    "注文数": [20, 15]
})

order_C = pd.DataFrame({
    "野菜": ["にんじん", "玉ねぎ"],
    "注文数": [10, 25]
})

order_D = pd.DataFrame({
    "野菜": ["キャベツ", "にんじん"],
    "注文数": [15, 10]
})

# 保存
order_A.to_excel(
    "project/samples/order_new/order_A_20230524.xlsx",
    index=False
)

order_B.to_excel(
    "project/samples/order_new/order_B_20230524.xlsx",
    index=False
)

order_C.to_excel(
    "project/samples/order_new/order_C_20230524.xlsx",
    index=False
)

order_D.to_excel(
    "project/samples/order_new/order_D_20230524.xlsx",
    index=False
)

# =========================================
# inventory.xlsx 作成
# =========================================

inventory_df = pd.DataFrame({
    "日付": ["2023-05-23"],
    "キャベツ": [100],
    "にんじん": [80],
    "玉ねぎ": [50]
})

inventory_df.to_excel(
    "project/samples/inventory.xlsx",
    index=False
)

# =========================================
# pickup.xlsx 作成
# =========================================

pickup_df = pd.DataFrame({
    "野菜": ["キャベツ", "にんじん", "玉ねぎ"],
    "しきい値": [20, 10, 15],
    "発注数": [100, 50, 80]
})

pickup_df.to_excel(
    "project/samples/pickup.xlsx",
    index=False
)

# =========================================
# 注文データ取得
# =========================================

order_dir = "project/samples/order_new"

all_data = []

for file_name in os.listdir(order_dir):

    if file_name.endswith(".xlsx"):

        file_path = os.path.join(order_dir, file_name)

        df = pd.read_excel(file_path)

        all_data.append(df)

# =========================================
# 注文集計
# =========================================

merged_df = pd.concat(all_data)

total_orders = merged_df.groupby("野菜")["注文数"].sum()

print("========== 合計注文数 ==========")
print(total_orders)

# =========================================
# 最新在庫取得
# =========================================

inventory_path = "project/samples/inventory.xlsx"

inventory_df = pd.read_excel(inventory_path)

latest_inventory = inventory_df.iloc[-1]

print("\n========== 最新在庫 ==========")
print(latest_inventory)

# =========================================
# 在庫差分計算
# =========================================

current_stock = {}

for vegetable in total_orders.index:

    stock = latest_inventory.get(vegetable, 0)

    order_qty = total_orders[vegetable]

    remaining = stock - order_qty

    current_stock[vegetable] = remaining

print("\n========== 出荷後在庫 ==========")
print(current_stock)

# =========================================
# 発注条件確認
# =========================================

pickup_path = "project/samples/pickup.xlsx"

pickup_df = pd.read_excel(pickup_path)

order_targets = []

for _, row in pickup_df.iterrows():

    vegetable = row["野菜"]

    threshold = row["しきい値"]

    order_amount = row["発注数"]

    stock = current_stock.get(vegetable, 0)

    if stock < threshold:

        order_targets.append({
            "野菜": vegetable,
            "現在在庫": stock,
            "発注数": order_amount
        })

# =========================================
# 発注メール本文作成
# =========================================

if order_targets:

    body = "以下の商品を発注お願いします。\n\n"

    for item in order_targets:

        body += (
            f"野菜: {item['野菜']}\n"
            f"現在在庫: {item['現在在庫']}\n"
            f"発注数: {item['発注数']}\n\n"
        )

    print("\n========== 発注メール ==========")
    print(body)

else:

    print("\n発注対象なし")

# =========================================
# inventory.xlsx 更新
# =========================================

new_row = {
    "日付": datetime.today().strftime("%Y-%m-%d")
}

for vegetable in inventory_df.columns[1:]:

    latest_stock = latest_inventory.get(vegetable, 0)

    order_qty = total_orders.get(vegetable, 0)

    new_stock = latest_stock - order_qty

    new_row[vegetable] = new_stock

inventory_df = pd.concat(
    [inventory_df, pd.DataFrame([new_row])],
    ignore_index=True
)

inventory_df.to_excel(
    inventory_path,
    index=False,
    engine="openpyxl"
)

print("\n========== inventory更新完了 ==========")

# =========================================
# 更新後在庫表示
# =========================================

updated_inventory = pd.read_excel(inventory_path)

print("\n========== 更新後 inventory.xlsx ==========")
print(updated_inventory)

print("\n========== 全処理終了 ==========")

========== 合計注文数 ==========
野菜
にんじん    40
キャベツ    65
玉ねぎ     40
Name: 注文数, dtype: int64

========== 最新在庫 ==========
日付      2023-05-23
キャベツ           100
にんじん            80
玉ねぎ             50
Name: 0, dtype: object

========== 出荷後在庫 ==========
{'にんじん': np.int64(40), 'キャベツ': np.int64(35), '玉ねぎ': np.int64(10)}

========== 発注メール ==========
以下の商品を発注お願いします。

野菜: 玉ねぎ
現在在庫: 10
発注数: 80



========== inventory更新完了 ==========

========== 更新後 inventory.xlsx ==========
           日付  キャベツ  にんじん  玉ねぎ
0  2023-05-23   100    80   50
1  2026-05-02    35    40   10

========== 全処理終了 ==========
